In [ ]:
import jax
import jax.numpy as jnp
import optax
import time

from qst_tec.gdchol_rank import custom_ppt_penalty, custom_nuclear_norm
from qst_tec.basicfunc import build_separable_state, distance_sq, partial_transpose

/Users/zwh/vscodeprojects/GD-QST_calculate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# ==========================================
# 第1部分：verifier section

In [ ]:
# 1. 验证器 Loss：只求最小化几何距离
@jax.jit
def verifier_loss(params_V, rho_target):
    A, B = params_V
    rho_sep = build_separable_state(A, B)
    return distance_sq(rho_target, rho_sep)

In [4]:
# ==========================================
# 第二部分：定义双方的 Loss 函数

In [ ]:
# ==========================================
# 2. 生成器 Loss：保持 PPT 的同时，最大化几何距离
@jax.jit
def generator_loss(T, params_V, lamb):
    # 构造当前目标态
    t_dag_t = jnp.conj(T.T) @ T
    rho_estimated = t_dag_t / jnp.trace(t_dag_t)
    
    # PPT 惩罚
    # 再次强调：一定要用修复了自定义 VJP 的版本！

    rho_pt = partial_transpose(rho_estimated)
    ppt_pen = custom_ppt_penalty(rho_pt)

    dist = verifier_loss(params_V, rho_estimated)
    
    # Min-Max 核心：生成器希望 loss 越小越好。
    # 所以它需要极小的 PPT 惩罚，和极大的 dist (所以是减去 dist)
    return lamb * ppt_pen - dist, (ppt_pen, dist)

In [6]:
# ==========================================
# 第三部分：JIT 编译双方的更新步 (交替更新)

In [7]:
# ==========================================

# 初始化两个优化器
# 通常在 GAN 中，验证器（判别器）学习率要快一些，保证它总是能追上生成器
lr_G = 0.01
lr_V = 0.05
opt_G = optax.adam(lr_G)
opt_V = optax.adam(lr_V)

@jax.jit
def update_verifier(params_V, opt_state_V, rho_target):
    loss_val, grads = jax.value_and_grad(verifier_loss)(params_V, rho_target)
    # JAX 复数梯度坑：对 Tuple 里的每一个矩阵取共轭
    grads = jax.tree.map(lambda g: jnp.conj(g), grads)
    updates, new_opt_state = opt_V.update(grads, opt_state_V, params_V)
    new_params = optax.apply_updates(params_V, updates)
    return new_params, new_opt_state, loss_val

@jax.jit
def update_generator(T, opt_state_G, params_V, lamb):
    (loss_val, (ppt, dist)), grads = jax.value_and_grad(generator_loss, has_aux=True)(T, params_V, lamb)
    grads = jnp.conj(grads) # JAX 复数梯度共轭
    updates, new_opt_state = opt_G.update(grads, opt_state_G, T)
    new_T = optax.apply_updates(T, updates)
    return new_T, new_opt_state, loss_val, ppt, dist

In [8]:
# ==========================================
# 第四部分：开始对抗训练 (The Game)

In [9]:
# ==========================================

def train_quantum_gan(d=4, K=None, rank_T=5, steps=5000, lamb=100.0):
    # Carathéodory bound: K_max = rank_T² is the theoretical maximum
    # for pure product states in an R-dimensional subspace.
    if K is None:
        K = rank_T ** 2

    key = jax.random.PRNGKey(42)
    key, k1, k2, k3, k4 = jax.random.split(key, 5)

    # 随机初始化双方参数
    T = jax.random.normal(k1, (rank_T, d**2)) + 1j * jax.random.normal(k2, (rank_T, d**2))
    A = jax.random.normal(k3, (K, d)) + 1j * jax.random.normal(k4, (K, d))
    B = jax.random.normal(k4, (K, d)) + 1j * jax.random.normal(k3, (K, d))
    params_V = (A, B)
    
    opt_state_G = opt_G.init(T)
    opt_state_V = opt_V.init(params_V)
    
    print(f"开始对抗搜索: d={d}, K={K} (Carathéodory bound = rank_T² = {rank_T**2}), 目标态秩={rank_T}")
    start_time = time.time()
    
    for i in range(steps):
        # --- 计算当前目标态 (作为验证器的靶子) ---
        t_dag_t = jnp.conj(T.T) @ T
        rho_target = t_dag_t / jnp.trace(t_dag_t)
        
        # --- 步骤 A：训练验证器 (让它充分拟合) ---
        # GAN 的经典策略：判别器多跑几步，保证评估准确
        for _ in range(5): 
            params_V, opt_state_V, v_loss = update_verifier(params_V, opt_state_V, rho_target)
            
        # --- 步骤 B：训练生成器 (让它逃脱拟合) ---
        T, opt_state_G, g_loss, ppt, dist = update_generator(T, opt_state_G, params_V, lamb)
        
        if i % 500 == 0:
            print(f"Step {i:4d} | 生成器逃逸距离: {dist:.5f} | 验证器追赶误差: {v_loss:.5f} | PPT惩罚: {ppt:.2e}")
            
    print(f"耗时: {time.time() - start_time:.2f}s")
    return T, params_V


# 配置使用双精度
jax.config.update("jax_enable_x64", True)
# K is auto-computed from rank_T via the Carathéodory bound (K = rank_T²).
final_T, final_V = train_quantum_gan(d=8, rank_T=32, steps=5000, lamb=50.0)

# 在训练结束后提取最终的态
t_dag_t = jnp.conj(final_T.T) @ final_T
rho_final = t_dag_t / jnp.trace(t_dag_t)

# 打印它的 CCNR 看看！
print("最终态的 CCNR 值:", custom_nuclear_norm(rho_final))

开始对抗搜索: d=8, K=1024 (Carathéodory bound = rank_T² = 1024), 目标态秩=32
Step    0 | 生成器逃逸距离: 0.03580 | 验证器追赶误差: 0.03768 | PPT惩罚: 3.47e-03
Step  500 | 生成器逃逸距离: 0.00182 | 验证器追赶误差: 0.00182 | PPT惩罚: 3.88e-07
Step 1000 | 生成器逃逸距离: 0.00269 | 验证器追赶误差: 0.00269 | PPT惩罚: 7.83e-07
Step 1500 | 生成器逃逸距离: 0.00346 | 验证器追赶误差: 0.00346 | PPT惩罚: 1.05e-06
Step 2000 | 生成器逃逸距离: 0.00411 | 验证器追赶误差: 0.00411 | PPT惩罚: 1.32e-06
Step 2500 | 生成器逃逸距离: 0.00465 | 验证器追赶误差: 0.00465 | PPT惩罚: 1.52e-06
Step 3000 | 生成器逃逸距离: 0.00500 | 验证器追赶误差: 0.00500 | PPT惩罚: 1.88e-06
Step 3500 | 生成器逃逸距离: 0.00525 | 验证器追赶误差: 0.00525 | PPT惩罚: 1.83e-06
Step 4000 | 生成器逃逸距离: 0.00544 | 验证器追赶误差: 0.00544 | PPT惩罚: 1.97e-06
Step 4500 | 生成器逃逸距离: 0.00563 | 验证器追赶误差: 0.00563 | PPT惩罚: 2.00e-06
耗时: 463.21s
最终态的 CCNR 值: 1.0000000000000002
